# Pass Receiver Prediction — full pipeline (Colab)

Upload the project folder to Drive, or clone it, then run the cells in order.


In [ ]:
!pip install floodlight torch -q

### If Colab shows a RESTART SESSION button, click it

floodlight needs `numpy>=2.1` and `pandas>=2.2`. If pip upgraded either one,
the copy already loaded in memory is stale. Restart, then **skip the install cell**.

In [ ]:
# Put the project and the downloaded data on Drive so nothing is lost on restart.
import os, shutil
from google.colab import drive
drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/idsse-pass-receiver'   # upload the folder here
os.makedirs(PROJECT, exist_ok=True)
%cd $PROJECT
print(sorted(os.listdir('.')))

In [ ]:
# floodlight downloads into its own package folder, which Colab wipes on restart.
# Point that folder at Drive so the ~350 MB download happens only once.
from floodlight.io.datasets import DATA_DIR
DRIVE_DATA = f'{PROJECT}/.raw'
os.makedirs(DRIVE_DATA, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
link = os.path.join(DATA_DIR, 'idsse_dataset')
if os.path.islink(link): os.remove(link)
elif os.path.isdir(link): shutil.rmtree(link)
os.symlink(DRIVE_DATA, link)
print('raw data ->', DRIVE_DATA, '|', len(os.listdir(DRIVE_DATA)), 'files already there')

## Optional: check the plumbing first

This fabricates fake matches with a known 1.6 s offset and runs everything on them.
Every baseline should score about 10% (the fake labels are random). If one scores
much higher, something is leaking. **Clear `work/` afterwards.**

In [ ]:
!python _selftest.py && python step2_passes.py && python step3_sync.py

In [ ]:
!rm -rf work figures   # clear the fake results before using real data

## Real data

Step 1 downloads and parses the first match. Parsing takes a few minutes
because the position XML has one element per player per frame.

In [ ]:
!python step1_load.py J03WMX

Steps 2 to 5 build the dataset. Step 4 downloads and parses all seven matches,
so the first run of it is the slow one (about 2.4 GB in total).

In [ ]:
!python step2_passes.py

In [ ]:
!python step3_sync.py

In [ ]:
!python step4_clean.py

In [ ]:
!python step5_dataset.py

## Explore, then baselines before any model

In [ ]:
!python step6_explore.py

In [ ]:
from IPython.display import Image, display
import glob
for f in sorted(glob.glob('figures/0[1-6]*.png')):
    print(f); display(Image(f))

In [ ]:
!python step7_baselines.py

## Model

GPU is optional — the model is small. Runtime > Change runtime type > T4 if you want it.

In [ ]:
!python step8_model.py     # shape + permutation-invariance check

In [ ]:
!python step9_train.py

In [ ]:
!python step10_analysis.py

In [ ]:
for f in sorted(glob.glob('figures/[01][0-9]_*.png')):
    print(f); display(Image(f))

## Results side by side

In [ ]:
import pandas as pd
print(pd.read_csv('work/baseline_results.csv').to_string(index=False))
print()
m = pd.read_csv('work/model_results.csv')
print(m.groupby(['test_match','setting'])[['top1','top3']].agg(['mean','min','max']).to_string())